# 03 - KPI Analysis
Calculates the four project KPIs (Network Stability Index, Latency Sensitivity Score, 
Packet Loss Impact Ratio, Network-Efficiency Threshold) using reusable functions from src/kpi.py.

Note: prior EDA and modeling found no statistically significant relationship between 
network variables and Efficiency_Status. These KPIs are reported as descriptive summaries 
of network conditions, not as evidence of network impact on efficiency.

In [2]:
import pandas as pd
import sys
sys.path.append('../src')

df = pd.read_csv('../data/raw/Thales_Group_Manufacturing.csv')
df.head()

,Date,Timestamp,Machine_ID,Operation_Mode,Temperature_C,Vibration_Hz,Power_Consumption_kW,Network_Latency_ms,Packet_Loss_%,Quality_Control_Defect_Rate_%,Production_Speed_units_per_hr,Predictive_Maintenance_Score,Error_Rate_%,Efficiency_Status
0,01-01-2025,00:00:00,39,Idle,74.138,3.501,8.612,10.651,0.208,7.751,477.657,0.345,14.965,Low
1,01-01-2025,00:01:00,29,Active,84.265,3.356,2.269,29.112,2.228,4.989,398.175,0.770,7.678,Low
2,01-01-2025,00:02:00,15,Active,44.280,2.080,6.144,18.357,1.639,0.457,108.075,0.987,8.198,Low
3,01-01-2025,00:03:00,43,Active,40.569,0.298,4.068,29.154,1.161,4.583,329.579,0.983,2.741,Medium
4,01-01-2025,00:04:00,8,Idle,75.064,0.346,6.226,34.029,4.797,2.288,159.114,0.573,12.101,Low


## Setup
Load data and reconstruct the Latency_Band / PacketLoss_Band columns needed by the KPI functions.

In [3]:
import sys
sys.path.append('../src')
from kpi import network_stability_index, packet_loss_impact_ratio

df['Latency_Band'] = pd.qcut(df['Network_Latency_ms'], 3, labels=['Low', 'Medium', 'High'])
df['PacketLoss_Band'] = pd.qcut(df['Packet_Loss_%'], 3, labels=['Low', 'Medium', 'High'])

df['Network_Stability_Index'] = network_stability_index(df)
print(df['Network_Stability_Index'].describe())

print(packet_loss_impact_ratio(df))

count    100000.000000
mean         49.990939
std          20.323804
min           0.054490
25%          35.471735
50%          49.999184
75%          64.496173
max          99.852245
Name: Network_Stability_Index, dtype: float64
{'defect_rate_change_%': np.float64(-0.6395988642025588), 'error_rate_change_%': np.float64(-0.2817987821134014)}


**Observation:** Network Stability Index averages ~50 with a roughly even spread (25th-75th percentile: 35-64), consistent with the underlying latency/packet-loss variables being independently distributed rather than clustered around specific "problem" conditions.

Packet Loss Impact Ratio shows a -0.64% change in defect rate and -0.28% change in error rate between low- and high-packet-loss periods — negligible in magnitude, and in the opposite direction from what the brief's hypothesis would predict. This is consistent with the near-zero correlations, non-significant chi-square tests, and low-recall classifier results found earlier: packet loss shows no meaningful impact on quality metrics in this dataset.